# Week 4 Exercise – Code Documentation & Unit Test Generator

**GOlivierNation** — End-of-week 4 contribution.

A single Gradio app that:
- **Docstrings:** Adds PEP-257 style docstrings to Python code (Google or NumPy style).
- **Unit tests:** Generates pytest-style unit tests for the given Python code.

Uses OpenRouter (gpt-4o-mini) or OpenAI; no changes outside `week4/community-contributions/GOlivierNation/`.

In [ ]:
# Imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Config: OpenRouter (primary) and OpenAI fallback
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
MODEL_OPENROUTER = "openai/gpt-4o-mini"
MODEL_OPENAI = "gpt-4o-mini"

if OPENROUTER_API_KEY:
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE)
    MODEL = MODEL_OPENROUTER
    print("Using OpenRouter.")
elif OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    MODEL = MODEL_OPENAI
    print("Using OpenAI.")
else:
    client = OpenAI()
    MODEL = MODEL_OPENAI
    print("Using default client; set OPENROUTER_API_KEY or OPENAI_API_KEY in .env.")

In [ ]:
# System prompts for docstrings and unit tests
DOCSTRING_SYSTEM = """You are a Python documentation expert. Given Python code, return only the same code with added or improved docstrings (PEP 257). Use the requested style (Google or NumPy). Do not add any text outside the code. Output only valid Python."""

UNITTEST_SYSTEM = """You are a testing expert. Given Python code, generate pytest-style unit tests. Return only the test code (imports and test functions). Use the requested style (Google or NumPy) for docstrings in tests. Output only valid Python."""

In [ ]:
def generate_docstrings(code: str, style: str) -> str:
    if not (code or code.strip()):
        return ""
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": DOCSTRING_SYSTEM},
                {"role": "user", "content": f"Style: {style}.\n\nPython code:\n{code}"},
            ],
        )
        return (r.choices[0].message.content or "").strip()
    except Exception as e:
        return f"Error: {e}"

def generate_unittests(code: str, style: str) -> str:
    if not (code or code.strip()):
        return ""
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": UNITTEST_SYSTEM},
                {"role": "user", "content": f"Style for test docstrings: {style}.\n\nPython code to test:\n{code}"},
            ],
        )
        return (r.choices[0].message.content or "").strip()
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# Gradio UI: Docstrings and Unit Tests (Week 4 only)
with gr.Blocks(title="Week 4 – Docstrings & Unit Tests", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Week 4 Exercise: Code Documentation & Unit Test Generator")
    style = gr.Dropdown(["Google", "NumPy"], value="Google", label="Docstring style")
    code_in = gr.Textbox(label="Python code", lines=14, placeholder="Paste Python code here...")
    with gr.Row():
        doc_btn = gr.Button("Add docstrings")
        test_btn = gr.Button("Generate unit tests")
    doc_out = gr.Textbox(label="Code with docstrings", lines=14)
    test_out = gr.Textbox(label="Generated unit tests", lines=14)
    doc_btn.click(fn=lambda c, s: generate_docstrings(c, s), inputs=[code_in, style], outputs=doc_out)
    test_btn.click(fn=lambda c, s: generate_unittests(c, s), inputs=[code_in, style], outputs=test_out)
demo.launch(inbrowser=True)